In [0]:
from pyspark.sql import functions as F

df = spark.read.table("dbr_dev_ua5816bd.team_tristar_bronze.stop_times")

df = df.withColumn(
    'trip_id',
    F.when(
        F.trim(F.col('trip_id')) == '',
        None
    ).otherwise(
        F.trim(F.col('trip_id'))
    )
)

df = df.withColumn(
    'arrival_time',
    F.when(
        ~F.col('arrival_time').rlike(r'^\d{1,2}:\d{2}:\d{2}$'),
        None
    ).otherwise(
        F.col('arrival_time')
    )
)

df = df.withColumn(
    'departure_time',
    F.when(
        ~F.col('departure_time').rlike(r'^\d{1,2}:\d{2}:\d{2}$'),
        None
    ).otherwise(
        F.col('departure_time')
    )
)

df = df.withColumn(
    'stop_id',
    F.when(
        F.col('stop_id').try_cast('int') <= 0,
        None
    ).otherwise(
        F.col('stop_id').try_cast('int')
    )
)

df = df.withColumn(
    'stop_sequence',
    F.when(
        F.col('stop_sequence').try_cast('int') <= 0,
        None
    ).otherwise(
        F.col('stop_sequence').try_cast('int')
    )
)

df = df.withColumn(
    'pickup_type',
    F.when(
        ~F.col('pickup_type').try_cast('int').isin(0, 1, 2, 3),
        None
    ).otherwise(
        F.col('pickup_type').try_cast('int')
    )
)

df = df.withColumn(
    'drop_off_type',
    F.when(
        ~F.col('drop_off_type').try_cast('int').isin(0, 1, 2, 3),
        None
    ).otherwise(
        F.col('drop_off_type').try_cast('int')
    )
)

df = df.withColumn(
    'stop_headsign',
    F.when(
        F.trim(F.col('stop_headsign')) == '',
        None
    ).otherwise(
        F.trim(F.col('stop_headsign'))
    )
)

df = df.withColumn(
    'source',
    F.when(
        F.trim(F.col('source')) == '',
        None
    ).otherwise(
        F.trim(F.col('source'))
    )
)

df = df.withColumn(
    'source_update_date',
    F.when(
        F.trim(F.col('source_update_date')) == '',
        None
    ).otherwise(
        F.trim(F.col('source_update_date'))
    )
)

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark,
    "dbr_dev_ua5816bd.team_tristar_silver.stop_times"
)

silver_table.alias("silver").merge(
    df.alias("bronze"),
    """
    silver.trip_id = bronze.trip_id
    AND silver.stop_sequence = bronze.stop_sequence
    """
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).whenNotMatchedBySourceDelete(
).execute()